# Zonal Statistics

**Series:** Raster Analysis in Python (notebook 2 of 2)
**Time:** about 30 minutes

**In this notebook you will:**
- Summarize a raster inside each polygon (zonal statistics)
- Compute mean elevation and percent developed land for every census tract
- Relate land cover to population density
- Use land cover to place people more realistically (dasymetric estimation)

**You should already know:** notebook 02a, and the Geoprocessing in Python episode (notebooks 01a to 01c).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## Rasters meet polygons

**Zonal statistics** summarize the raster cells that fall inside each zone (a polygon): the mean, minimum,
maximum or sum for measurements such as elevation, or the count of each class for categories such as land
cover. It is how raster data gets into a table of tracts, counties or neighborhoods, where it can be joined to
Census data.

We use the **rasterstats** library, which is not part of Colab, so we install it first.

In [ ]:
!pip install -q rasterstats

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rasterstats import zonal_stats

DEM = "https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_dem.tif"
LC = "https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_landcover_2021.tif"
tracts = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_tracts_acs.geojson").to_crs(epsg=26916)   # the rasters' CRS
print(len(tracts), "tracts")

The polygons must be in the **same CRS as the raster**, here EPSG:26916, or they will miss it completely.

## Step 1: Mean elevation per tract

In [ ]:
stats = zonal_stats(tracts, DEM, stats=["mean", "min", "max"])
tracts["elev_mean"] = [s["mean"] for s in stats]
tracts["elev_range"] = [s["max"] - s["min"] for s in stats]
tracts[["name", "elev_mean", "elev_range"]].sort_values("elev_mean").head().round(1)

## Step 2: Land cover counts per tract

For a categorical raster, `categorical=True` returns a dictionary for each polygon: `{class code: number of
cells}`. Averaging codes would be meaningless.

In [ ]:
counts = zonal_stats(tracts, LC, categorical=True, nodata=0)
print(counts[0])                                   # the first tract

DEVELOPED = [21, 22, 23, 24]
tracts["cells"] = [sum(c.values()) for c in counts]
tracts["dev_cells"] = [sum(n for code, n in c.items() if code in DEVELOPED) for c in counts]
tracts["pct_developed"] = tracts["dev_cells"] / tracts["cells"] * 100
tracts["pct_developed"].describe().round(1)

A quick check: the cells counted should cover each tract's area (each cell is 900 m²). Small differences come
from cells on the boundary: rasterstats counts a cell if its **center** is inside the polygon.

In [ ]:
ratio = tracts["cells"] * 900 / tracts.area
print(ratio.round(2).min(), "to", ratio.round(2).max())

## Step 3: Land cover and population density

In [ ]:
tracts["density"] = tracts["total_pop"] / (tracts.area / 1e6)           # people per km2

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(tracts["pct_developed"], tracts["density"], color="tab:red")
ax.set_yscale("log")
ax.set_xlabel("Developed land (% of tract)")
ax.set_ylabel("People per km2 (log scale)")
plt.show()
r = np.corrcoef(tracts["pct_developed"], np.log(tracts["density"]))[0, 1]
print("Correlation with log density:", round(r, 2))

Developed land and population density go together closely. That is the idea behind **dasymetric mapping**:
use land cover to decide *where inside a tract* people live, instead of assuming they are spread evenly (the
areal weighting of the Geoprocessing episode).

## Step 4: Dasymetric estimate versus areal weighting

In Geoprocessing notebook 01b, areal weighting estimated about **114,000** people in the two cities and about
**77,500** within 1 mile of a supermarket. Now we redo it, giving each piece of a tract a share of its people
in proportion to its **developed cells** rather than its area.

In [ ]:
stores = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/grocery_stores.geojson").to_crs(epsg=26916)
city = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/study_area.geojson").to_crs(epsg=26916)
supermarkets = stores[stores["store_type"] == "Supermarket"]
zone = gpd.GeoDataFrame(geometry=[supermarkets.buffer(1609).union_all()], crs=26916)

tracts["tract_m2"] = tracts.area
in_city = gpd.clip(tracts, city)
near = gpd.overlay(in_city, zone, how="intersection")


def developed_cells(polygons):
    return [sum(n for code, n in c.items() if code in DEVELOPED)
            for c in zonal_stats(polygons, LC, categorical=True, nodata=0)]


for pieces in (in_city, near):
    pieces["pop_areal"] = pieces.area / pieces["tract_m2"] * pieces["total_pop"]
    pieces["pop_dasy"] = np.array(developed_cells(pieces)) / pieces["dev_cells"] * pieces["total_pop"]

summary = pd.DataFrame({
    "areal weighting": [in_city["pop_areal"].sum(), near["pop_areal"].sum()],
    "dasymetric": [in_city["pop_dasy"].sum(), near["pop_dasy"].sum()],
}, index=["in the cities", "within 1 mile of a supermarket"]).round(0)
summary.loc["share near"] = (summary.iloc[1] / summary.iloc[0] * 100).round(1)
summary

The dasymetric method puts **more** people inside the cities: the tracts on the city edge are partly farmland,
and areal weighting wrongly placed some of their residents in the fields outside the limits. Yet the **share**
within 1 mile of a supermarket hardly changes. A method can change counts a lot and conclusions a little, or the
reverse; comparing two methods is a cheap way to see how much your answer depends on an assumption.

Dasymetric estimates still assume people are spread evenly over *developed* land, which includes parking lots,
roads and shopping centers. Weighting low- and high-intensity developed classes differently, or using building
footprints, refines it further.

## Try it yourself

Which tract has the **largest share of forest** (NLCD codes 41, 42 and 43)? Compute `pct_forest` the same way
as `pct_developed` and sort.

<details>
<summary><b>Show a solution</b></summary>

```python
FOREST = [41, 42, 43]
tracts["pct_forest"] = [sum(n for code, n in c.items() if code in FOREST) / sum(c.values()) * 100
                        for c in counts]
tracts.sort_values("pct_forest", ascending=False)[["name", "pct_forest"]].head(3).round(1)
```

</details>

In [ ]:
# Write your answer here

## Recap

- Zonal statistics summarize raster cells inside polygons; put the polygons in the raster's CRS first
- Use mean, min, max for measurements and class counts (`categorical=True`) for categories
- A cell counts if its center is inside the polygon, so small polygons are approximate
- Dasymetric estimation uses land cover to place people; compare it with areal weighting

**Next:** the advanced episodes start with Network Analysis (notebooks 03a to 03c). The Case Studies module's park
access case study uses these tools on a real question.